# Laboratory work 4

### Task 1. Compute the moving average of window size 3, for the given 1D array.

In [ ]:
import numpy as np
np.random.seed(100)
Z = np.random.randint(10, size=10)
# array:  [8 8 3 7 7 0 4 2 5 2]
# moving average:  [ 6.33  6.    5.67  4.67  3.67  2.    3.67  3.  ]

# Simple moving average using convolution
"""averages = np.convolve(Z, np.ones(3)/3, mode='valid')
print(averages)"""

moving_average = [
    # Calculate the mean of 3 consecutive elements in Z
    np.mean(Z[i:i + 3])
    # Z - 2 because we are taking the average of 3 elements at a time 
    # (we need to stop 2 elements before the end of the array)
    for i in range(len(Z) - 2) 
    # An array of length n has n - 3 + 1 windows of size 3, so the length of the moving average array is n - 2
]



print(np.round(moving_average, 2))

[6.33333333 6.         5.66666667 4.66666667 3.66666667 2.
 3.66666667 3.        ]


### Task 2. Find the index of 5th repetition of number 1 in x.

In [1]:
import numpy as np
x = np.array([1, 2, 1, 1, 3, 4, 3, 1, 1, 2, 1, 1, 2])

print(f"The array size: {x.size}")

# Ask user to input a target value to count in the array and index of the target value
target_value = int(input("Enter a target value to count in the array: "))
target_occurence = int(input("Enter the target occurrence number to find the index of: "))

# use dict to store the counts of each unique value in x
counts = {target_value: 0}

for i, value in enumerate(x):
    if value == target_value:
        counts[target_value] += 1

        if counts[target_value] == target_occurence:
            print(f"The target's value '{target_value}' {target_occurence}th repitiion is found at index {i}.")
            break

The array size: 13
The target's value '1' 5th repitiion is found at index 8.


In [3]:
# Simpler approach
import numpy as np
x = np.array([1, 2, 1, 1, 3, 4, 3, 1, 1, 2, 1, 1, 2])

print(f"The array size: {x.size}")

# Ask user to input a target value to count in the array and index of the target value
target_value = int(input("Enter a target value to count in the array: "))
target_occurence = int(input("Enter the target occurrence number to find the index of: "))

count = 0

for i, value in enumerate(x):
    if value == target_value:
        count += 1
        if count == target_occurence:
            print(
                f"The {target_occurence}th occurrence of {target_value} "
                f"is at index {i}."
            )
            break

The array size: 13
The 5th occurrence of 1 is at index 8.


### Task 3. Drop all nan values from a 1D numpy array

In [ ]:
arr = np.array([1,2,3,np.nan,5,6,7,np.nan])

# ~ is used to invert the boolean array returned by np.isnan(arr), so that we select only the non-NaN values from arr.
cleaned_array = arr[~np.isnan(arr)]  
print(cleaned_array)  # Output: [1. 2. 3. 5. 6. 7.]

[1. 2. 3. 5. 6. 7.]


### Task 4: Weather Data Analysis with NumPy I/O

#### Data generation
Generate synthetic temperature data for 365 days of the year for 5 cities (matrix shape (365, 5)), using random numbers around realistic averages.
Save the data in three formats:

- Binary NumPy format (.npy)

- Text file (.txt)

- CSV file (.csv)

#### Data loading
Load the .npy file back into NumPy.

#### Processing
For each city, compute: 

- Minimum, maximum, mean, and standard deviation of temperature.

- The hottest and coldest day (index of the day).

- Normalize the data (convert to z-scores).

#### Advanced requirement
- Find the day of the year when the most cities had above-average temperature.

- Create a boolean mask for days when all cities were below 0°C.

#### Saving results
Save the computed statistics (per city) into a CSV file with columns: 
City, Min, Max, Mean, Std, HottestDay, ColdestDay. Name it "stats.csv"

Save the normalized data into a .npy file.

In [5]:
import pandas as pd
import numpy as np

# Generate seasonal temperature data with daily variation.
np.random.seed(42) # Ensures that exact same sequence is generated
days = np.arange(365) # Creates a 1D NumPy array containing sequential integers from 0 to 364.
city_names = ["Almaty", "Astana", "Moscow", "Tashkent", "Bishkek"]
annual_means = np.array([8, 9, 10, 11, 12])
seasonal_amplitudes = np.array([12, 13, 14, 15, 16])

# This creates a smooth sinusoidal wave that repeats every 365 days to simulate summer/winter seasons
season = np.cos(2 * np.pi * (days - 200) / 365)  # Coldest around January
temperatures = (
    annual_means
    + season[:, None] * seasonal_amplitudes
    + np.random.normal(0, 3, size=(365, 5))
)

# Save in three formats.
np.save("temperatures.npy", temperatures)
np.savetxt("temperatures.txt", temperatures, fmt="%.2f")
np.savetxt("temperatures.csv", temperatures, delimiter=",", fmt="%.2f")

# Load the binary file and calculate per-city statistics.
temperatures = np.load("temperatures.npy")
means = temperatures.mean(axis=0)

stats = pd.DataFrame({
    "City": city_names,
    "Min": temperatures.min(axis=0),
    "Max": temperatures.max(axis=0),
    "Mean": means,
    "Std": temperatures.std(axis=0),
    "HottestDay": temperatures.argmax(axis=0),  # 0-based day index
    "ColdestDay": temperatures.argmin(axis=0),  # 0-based day index
})
stats.to_csv("stats.csv", index=False)

# Normalize each city's temperatures using its own mean and standard deviation.
normalized = (temperatures - means) / temperatures.std(axis=0)
np.save("temperatures_normalized.npy", normalized)

# Find the day with the most cities above their respective annual averages.
above_average_counts = (temperatures > means).sum(axis=1)
day_with_most_above_average = int(above_average_counts.argmax())

# Boolean mask: True for days when every city was below 0°C.
all_cities_below_zero = (temperatures < 0).all(axis=1)

print(f"Day with the most cities above average: {day_with_most_above_average}")
print(f"Number of such cities: {above_average_counts[day_with_most_above_average]}")
print("Days when all cities were below 0°C:", np.where(all_cities_below_zero)[0])
print(stats)

Day with the most cities above average: 114
Number of such cities: 5
Days when all cities were below 0°C: [  2   3   5   7   8   9  10  11  12  15  17  18  19  20  23  26  29  30
  32  34  36  37  38  39  43  52  55 355 356 358 361 363 364]
       City        Min        Max       Mean        Std  HottestDay  \
0    Almaty -10.280188  26.571169   8.000840   9.072031         176   
1    Astana  -9.729884  28.204561   8.999025   9.571244         223   
2    Moscow -11.326583  30.217486  10.137710  10.465272         202   
3  Tashkent  -9.209053  32.391266  11.151953  11.030139         204   
4   Bishkek -11.830204  33.641568  12.359173  12.051514         181   

   ColdestDay  
0         350  
1         363  
2          52  
3           2  
4          14  
